[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dilru1/microlensing-dtw/blob/main/notebooks/03_tE_recovery.ipynb)

# Step 3 — Does the distance keep the physics?

The Einstein time t_E sets the lens mass, so a good similarity must keep it.
We predict log₁₀ t_E of stellar and free-floating-planet events with **5-nearest-neighbour regression**.

- Removing the **shift** (t₀) should help.
- Allowing **stretch** (time warping) may destroy t_E.

Takes about 2 minutes.

In [ ]:
!pip -q install tslearn VBMicrolensing
!git clone -q https://github.com/Dilru1/microlensing-dtw.git
import sys; sys.path.insert(0, "microlensing-dtw")

import numpy as np
import matplotlib.pyplot as plt
from mldtw.simulate import make_dataset, CLASSES, T
from mldtw.bench import Bench
FIG_DIR = "microlensing-dtw/figures"

In [ ]:
Xtr, ytr, ptr = make_dataset(150, seed=1, classes=[2, 3])   # 300 events
Xte, yte, pte = make_dataset(50, seed=2, classes=[2, 3])     # 100 events
ttr = np.log10([p["tE"] for p in ptr])
tte = np.log10([p["tE"] for p in pte])

bench = Bench(Xtr, Xte).run()
rmse = bench.regress(ttr, tte)
rmse

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for a, n in zip(ax, ["Euclid, centred", "Centred + DTW"]):
    a.scatter(tte, bench.pred[n], c=yte, s=12, cmap="coolwarm")
    a.plot([-1.5, 2], [-1.5, 2], "k--"); a.set_title(f"{n}: {rmse[n]:.3f} dex")
    a.set_xlabel("true log10 tE [d]")
ax[0].set_ylabel("predicted log10 tE")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/step3_tE.png", dpi=150)

**Read the result**
- Euclid → Euclid, centred: the error from t₀ shifts alone.
- Euclid, centred → Centred + DTW: what free warping does to t_E.
- Centred + DTW band sits in between: a little warping is tolerable.

**Message:** for microlensing the right similarity is **shift-invariant but not stretch-invariant**. DTW can classify well (Step 2) while blurring the physical parameter we care about.